# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading, exploring, and analyzing the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. 

### Dataset Source
* Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Name:", metadata.name)
print("Description:", metadata.description)


## 2. Data Overview

Let's review the available record sets, fields, and their unique `@id`s (identifiers). This helps understand how the data is organized in the FAIR² Croissant package.

In [ ]:
# List all record sets and their fields, referencing by @id
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet: {rs['@id']} (name: {rs.get('name', '[unnamed]')})")
    if 'field' in rs:
        fields = rs['field']
        for field in fields:
            if isinstance(field, dict):
                print(f"  Field: {field['@id']} (name: {field.get('name','[unnamed]')})")
            else:
                print(f"  Field: {field}")
    print("-")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

# For demonstration, use the first record set (often the main tabular data set)
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"DataFrame loaded for RecordSet {record_set_id} with shape {dataframes[record_set_id].shape}")
    else:
        print(f"No records found for RecordSet {record_set_id}")

if dataframes:
    first_record_set = list(dataframes.keys())[0]
    print(f"\nFields (@id) in {first_record_set}:")
    print(dataframes[first_record_set].columns.tolist())
    display(dataframes[first_record_set].head())

## 4. Exploratory Data Analysis (EDA)

Apply typical data processing steps, such as filtering records based on a numeric field, normalizing numeric columns, and grouping by a key attribute. All fields must be referenced by their `@id` only.

In [ ]:
# Pick the main record set
main_record_set_id = first_record_set
df = dataframes[main_record_set_id].copy()

# Display columns with example values to help find numeric fields
print("Example values per column:")
for col in df.columns:
    print(f"{col}: {df[col].iloc[0]}")

# Find candidate numeric field (infer from column types and data)
numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])] 
# If none are automatically detected (e.g., all string), try to coerce
if not numeric_candidates:
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            if df[col].notna().any() and df[col].dtype != object:
                numeric_candidates.append(col)
        except Exception:
            pass

print("\nNumeric field candidates by @id:", numeric_candidates)

if numeric_candidates:
    numeric_field_id = numeric_candidates[0]  # Pick first as example
    # Filter: threshold = mean of field (or pick a value if suitable)
    threshold = df[numeric_field_id].dropna().mean() if df[numeric_field_id].notna().any() else 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    normalized_col = numeric_field_id + '_normalized'
    filtered_df[normalized_col] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, normalized_col]].head())

    # Look for possible group field (categorical with a low number of unique values)
    group_field_id = None
    for col in df.columns:
        nunique = df[col].nunique()
        if pd.api.types.is_string_dtype(df[col]) and nunique < 10 and col != numeric_field_id:
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped (mean) by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No suitable group field found.")
else:
    print("No numeric fields detected for EDA.")

## 5. Visualization

Visualize the distribution or relationship of fields. Below is an example histogram and, if possible, a grouped bar plot using the selected fields (always referenced by `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'numeric_field_id' in locals():
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if 'group_field_id' in locals() and group_field_id is not None:
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
    else:
        print("No suitable group field for group-wise plot.")
else:
    print("No numeric field to visualize.")

## 6. Conclusion

In this notebook, we:

* Loaded and examined the FAIR² colorectal cancer survivor dataset using `mlcroissant`.
* Explored the record set structure and systematically referenced fields using their Croissant `@id`.
* Loaded tabular data, inferred numeric fields for EDA, normalized and grouped data, and visualized distributions.

This workflow facilitates streamlined, reproducible exploration of clinical tabular datasets following data standards. For further analysis, continue referencing all entities by their precise `@id`, and consult both Croissant schema and dataset documentation for field semantics and analytical best practices.